In [ ]:
!pip install -q sentence-transformers

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
#%load_ext cuml.accel
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin

from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

In [ ]:
df= pd.read_csv("/kaggle/input/nlp-getting-started/train.csv")
df

In [ ]:
df.isnull().sum()

In [ ]:
X=df.drop(columns=["id","target"])
y=df["target"]

In [ ]:
class SBERTTransformer(BaseEstimator, TransformerMixin):
    _shared_model = None

    def __init__(self, model_name="sentence-transformers/all-MiniLM-L6-v2", show_progress_bar=False):
        self.model_name = model_name
        self.show_progress_bar = show_progress_bar

        if SBERTTransformer._shared_model is None:
            print(f"Loading SBERT model: {self.model_name}")
            SBERTTransformer._shared_model = SentenceTransformer(self.model_name)

        self.model = SBERTTransformer._shared_model

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        # Ensure X is 1D array of strings
        if isinstance(X, pd.DataFrame):
            X = X.squeeze()
        X_filled = np.where(pd.isna(X), "Nan", X)
        return np.array(
            self.model.encode(X_filled.tolist(), show_progress_bar=self.show_progress_bar)
        )

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('text_embed', SBERTTransformer(), 'text')
    ],
    remainder='drop'
)

In [ ]:
# pipelines = {
#     'SVM': Pipeline([
#         ('sbert', preprocessor),
#         ('scaler', StandardScaler()),
#         ('classifier', SVC(random_state=42, probability=True))  # CPU only
#     ]),
    
#     'Logistic Regression': Pipeline([
#         ('sbert', preprocessor),
#         ('scaler', StandardScaler()),
#         ('classifier', LogisticRegression(random_state=42, max_iter=1000))  # CPU only
#     ]),
    
#     'Random Forest': Pipeline([
#         ('sbert', preprocessor),
#         ('classifier', RandomForestClassifier(random_state=42))  # CPU only
#     ]),

#     'LGBMClassifier': Pipeline([
#         ('sbert', preprocessor),
#         ('classifier', LGBMClassifier(random_state=42, device='gpu'))  # GPU enabled
#     ]),

#     'KNeighborsClassifier': Pipeline([
#         ('sbert', preprocessor),
#         ('classifier', KNeighborsClassifier())  # CPU only
#     ]),

#     'XGBClassifier': Pipeline([
#         ('sbert', preprocessor),
#         ('classifier', XGBClassifier(random_state=42, tree_method='gpu_hist', predictor='gpu_predictor'))  # GPU enabled
#     ]),

#     'CatBoostClassifier': Pipeline([
#         ('sbert', preprocessor),
#         ('classifier', CatBoostClassifier(random_state=42, task_type='GPU', verbose=0))  # GPU enabled
#     ])
# }

In [ ]:
# for name, pipe in pipelines.items():
#     scores = cross_val_score(pipe, X, y, cv=5, scoring='f1')
#     print(f"{name} Mean F1: {scores.mean():.4f}")

In [ ]:
pipe=Pipeline([
        ('sbert', preprocessor),
        ('classifier', CatBoostClassifier(random_state=42, task_type='GPU', verbose=0))  # GPU enabled
    ])

In [ ]:
best_estimator=pipe.fit(X,y)

SVM Mean F1: 0.7423
Logistic Regression Mean F1: 0.7298
Random Forest Mean F1: 0.7158
LGBMClassifier Mean F1: 0.7324
KNeighborsClassifier Mean F1: 0.7160
XGBClassifier Mean F1: 0.7286
CatBoostClassifier Mean F1: 0.7425

+++++++++++++++++++++++++++++TEST+++++++++++++++++++++++++

In [ ]:
df = pd.read_csv("/kaggle/input/nlp-getting-started/test.csv")

In [ ]:
X=df.drop(columns=["id"])

In [ ]:
predictions = best_estimator.predict(X)

In [ ]:
submission = pd.DataFrame({
    "id": df["id"],
    "target": predictions
})

submission.to_csv("submission.csv", index=False)
print("Saved submission.csv for Kaggle")